# 셀1. 경로설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
STORE = BASE / "data/external/store"
OUT = BASE / "data/processed"
DOCS = BASE / "docs"

print(sorted(p.name for p in STORE.iterdir()))

# 셀2. 파일 구조 확인

In [ ]:
for f in sorted(STORE.glob("*.csv")):
    for enc in ["utf-8", "utf-8-sig", "cp949"]:
        try:
            df = pd.read_csv(f, encoding=enc, nrows=5, low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"=== {f.name} ({enc}) ===")
    print("컬럼:", df.columns.tolist())
    print(df.head(3).T.to_string())
    print()

# 셀3. 강남구, 춘천시 추출 + 좌표 변환 + 저장

In [ ]:
import unicodedata
from pyproj import Transformer

def find_file(keyword):
    for p in STORE.glob("*.csv"):
        if keyword in unicodedata.normalize("NFC", p.name):
            return p
    raise FileNotFoundError(keyword)

USECOLS = ["상가업소번호", "상호명", "상권업종대분류명", "상권업종중분류명", "상권업종소분류명",
           "표준산업분류명", "시도명", "시군구명", "행정동명", "도로명주소", "층정보", "경도", "위도"]

parts = []
for f, sgg, region in [("서울", "강남구", "강남"), ("강원", "춘천시", "춘천")]:
    path = find_file(f"_{f}_")
    print("읽는 파일:", unicodedata.normalize("NFC", path.name))
    df = pd.read_csv(path, encoding="utf-8", usecols=USECOLS, low_memory=False)
    df = df[df["시군구명"] == sgg].copy()
    df["region"] = region
    parts.append(df)
stores = pd.concat(parts, ignore_index=True)

tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
stores["x"], stores["y"] = tf.transform(stores["경도"].values, stores["위도"].values)

stores.to_csv(OUT / "stores_2506.csv", index=False, encoding="utf-8-sig")
print(stores.groupby("region").size())
print(stores.groupby(["region", "상권업종대분류명"]).size().unstack(0).fillna(0).astype(int))

# 셀 4. 업종 분류 목록 뽑기 (매칭표 재료)

In [ ]:
cat = (stores.groupby(["상권업종대분류명", "상권업종중분류명", "상권업종소분류명", "region"])
             .size().unstack("region", fill_value=0).reset_index())
cat = cat.sort_values(["상권업종대분류명", "상권업종중분류명", "상권업종소분류명"])
cat.to_csv(DOCS / "store_categories.csv", index=False, encoding="utf-8-sig")
print(f"소분류 {len(cat)}개")
print(cat.groupby("상권업종대분류명")[["강남", "춘천"]].sum())

# 셀 5. 강남·춘천 골프 점포 확인

In [ ]:
golf = stores[stores["상권업종소분류명"].str.contains("골프", na=False) |
              stores["상호명"].str.contains("골프", na=False)]
print(golf.groupby(["region", "상권업종소분류명"]).size().unstack(0, fill_value=0))
print()
for r in ["강남", "춘천"]:
    print(f"[{r}] 골프 점포 예시")
    print(golf[golf["region"] == r][["상호명", "상권업종소분류명", "층정보", "도로명주소"]]
          .head(10).to_string(index=False))
    print()

# 셀6. 매칭 적용

In [ ]:
mapping = pd.read_csv(DOCS / "store_mapping.csv")
stores = stores.merge(mapping[["상권업종소분류명", "ind_grp", "비고"]], on="상권업종소분류명", how="left")
print("미매칭:", stores["ind_grp"].isna().sum())

stores_use = stores[~stores["ind_grp"].str.startswith("제외")].copy()
stores_use.to_csv(OUT / "stores_mapped.csv", index=False, encoding="utf-8-sig")
print(stores_use.groupby(["ind_grp", "region"]).size().unstack(fill_value=0))